## Hard Drive Failure Analytics Analytics - Spark Development
Purpose: The purpose of this notebook to refactor ingestion code to utilize spark and setup batch processing and incorporate into ingest_data.py file
Objectives:
- Create spark session
- Extract zipfiles from source
- Uzip files and import CSV files
- Convert to parquet

In [1]:
import os
from pathlib import Path
import pandas as pd
import pyspark
from pyspark.sql.functions import date_format, to_date, datediff, col, unix_timestamp, max
from pyspark.sql import SparkSession
from pyspark.sql import types

In [3]:
# Initialize Spark Session
spark = SparkSession.builder \
    .master("local[4]") \
    .config("spark.ui.port", "4040") \
    .appName('hard_drive_spark_app') \
    .config("spark.driver.bindAddress", "localhost") \
    .config("spark.local.ip", "127.0.0.1") \
    .getOrCreate()

26/03/25 23:40:21 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [24]:
csv_dir = "../data/source_csv/2025/data_Q1_2025"

In [25]:
df_spark = spark.read \
    .option("header", "true") \
        .option("inferSchema", "true") \
            .csv(f'{csv_dir}')

In [26]:
df_spark.count()

27799986

In [27]:
df_spark.printSchema()

root
 |-- date: date (nullable = true)
 |-- serial_number: string (nullable = true)
 |-- model: string (nullable = true)
 |-- capacity_bytes: long (nullable = true)
 |-- failure: integer (nullable = true)
 |-- datacenter: string (nullable = true)
 |-- cluster_id: integer (nullable = true)
 |-- vault_id: integer (nullable = true)
 |-- pod_id: integer (nullable = true)
 |-- pod_slot_num: integer (nullable = true)
 |-- is_legacy_format: boolean (nullable = true)
 |-- smart_1_normalized: integer (nullable = true)
 |-- smart_1_raw: long (nullable = true)
 |-- smart_2_normalized: integer (nullable = true)
 |-- smart_2_raw: integer (nullable = true)
 |-- smart_3_normalized: integer (nullable = true)
 |-- smart_3_raw: integer (nullable = true)
 |-- smart_4_normalized: integer (nullable = true)
 |-- smart_4_raw: integer (nullable = true)
 |-- smart_5_normalized: integer (nullable = true)
 |-- smart_5_raw: integer (nullable = true)
 |-- smart_7_normalized: integer (nullable = true)
 |-- smart_7_

In [8]:
# pulling schema from pandas dataframe
first_file = f"{csv_dir}/{os.listdir(f'{csv_dir}')[0]}"
df_pd = pd.read_csv(f'{first_file}', nrows=1000)
df_pd.dtypes

date                        str
serial_number               str
model                       str
capacity_bytes            int64
failure                   int64
                         ...   
smart_252_raw           float64
smart_254_normalized    float64
smart_254_raw           float64
smart_255_normalized    float64
smart_255_raw           float64
Length: 197, dtype: object

In [9]:
dtype = {
    "serial_number": "string",
    "model": "string",
    "capacity_bytes": "Int64",
    "failure": "Int64",
    "datacenter": "string",
    "cluster_id": "Int64",
    "vault_id": "Int64",
    "pod_id": "Int64",
    "pod_slot_num": "float64",
    "is_legacy_format": "boolean",
    "smart_5_normalized": "float64",  # SMART attribute (normalized value)
    "smart_5_raw": "float64", # Raw SMART value
    "smart_187_normalized": "float64",
    "smart_187_raw": "float64",
    "smart_188_normalized": "float64",
    "smart_188_raw": "float64",
    "smart_197_normalized": "float64",
    "smart_197_raw": "float64",
    "smart_198_normalized": "float64",
    "smart_198_raw": "float64"
}

parse_dates = [
    "date"
]

In [10]:
col_to_keep = []
for col in df_pd.columns:
    if col in dtype or col in parse_dates:
        col_to_keep.append(col)
print(col_to_keep)

['date', 'serial_number', 'model', 'capacity_bytes', 'failure', 'datacenter', 'cluster_id', 'vault_id', 'pod_id', 'pod_slot_num', 'is_legacy_format', 'smart_5_normalized', 'smart_5_raw', 'smart_187_normalized', 'smart_187_raw', 'smart_188_normalized', 'smart_188_raw', 'smart_197_normalized', 'smart_197_raw', 'smart_198_normalized', 'smart_198_raw']


In [11]:
df_spark_schema = types.StructType([
    types.StructField("date", types.TimestampType(), True),
    types.StructField("serial_number", types.StringType(), True),
    types.StructField("model", types.StringType(), True),
    types.StructField("capacity_bytes", types.IntegerType(), True),
    types.StructField("failure", types.IntegerType(), True),
    types.StructField("datacenter", types.StringType(), True),
    types.StructField("cluster_id", types.IntegerType(), True),
    types.StructField("vault_id", types.IntegerType(), True),
    types.StructField("pod_id", types.IntegerType(), True),
    types.StructField("pod_slot_num", types.DoubleType(), True),
    types.StructField("is_legacy_format", types.BooleanType(), True),
    types.StructField("smart_5_normalized", types.DoubleType(), True),
    types.StructField("smart_5_raw", types.DoubleType(), True),
    types.StructField("smart_187_normalized", types.DoubleType(), True),
    types.StructField("smart_187_raw", types.DoubleType(), True),
    types.StructField("smart_188_normalized", types.DoubleType(), True),
    types.StructField("smart_188_raw", types.DoubleType(), True),
    types.StructField("smart_197_normalized", types.DoubleType(), True),
    types.StructField("smart_197_raw", types.DoubleType(), True),
    types.StructField("smart_198_normalized", types.DoubleType(), True),
    types.StructField("smart_198_raw", types.DoubleType(), True)
])

In [12]:
year = 2025
qtr = "1"
file_suffix = f"{"data_Q"+qtr}_{year}"

In [13]:
os.makedirs(f"../data/pq/{year}/{file_suffix}", exist_ok=True)

In [14]:
pq_output_dir = f"../data/pq/{year}/{file_suffix}"

In [15]:
df_spark_re = spark.read \
        .option("header", "true") \
        .schema(df_spark_schema) \
        .csv(f'{csv_dir}')

In [16]:
df_spark_re.count()

27799986

In [17]:
df_spark_re.show(5)

+-------------------+-------------+--------------------+--------------+-------+----------+----------+--------+------+------------+----------------+------------------+-----------+--------------------+-------------+--------------------+-------------+--------------------+-------------+--------------------+-------------+
|               date|serial_number|               model|capacity_bytes|failure|datacenter|cluster_id|vault_id|pod_id|pod_slot_num|is_legacy_format|smart_5_normalized|smart_5_raw|smart_187_normalized|smart_187_raw|smart_188_normalized|smart_188_raw|smart_197_normalized|smart_197_raw|smart_198_normalized|smart_198_raw|
+-------------------+-------------+--------------------+--------------+-------+----------+----------+--------+------+------------+----------------+------------------+-----------+--------------------+-------------+--------------------+-------------+--------------------+-------------+--------------------+-------------+
|2025-03-09 00:00:00| 2207E60CC65A|      CT

26/03/25 23:42:11 WARN CSVHeaderChecker: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 197, schema size: 21
CSV file: file:///home/user1129/MyDocuments/10_14_Life_Admin/13_Technology/100_Coding/100_01_Python/data_engineering/hard_drive_failure_analytics_dashboard/data/source_csv/2025/data_Q1_2025/2025-03-09.csv


In [19]:
df_spark_re.repartition(32).write.parquet(pq_output_dir, mode="overwrite")

26/03/25 23:45:35 WARN CSVHeaderChecker: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 197, schema size: 21
CSV file: file:///home/user1129/MyDocuments/10_14_Life_Admin/13_Technology/100_Coding/100_01_Python/data_engineering/hard_drive_failure_analytics_dashboard/data/source_csv/2025/data_Q1_2025/2025-03-12.csv
26/03/25 23:45:35 WARN CSVHeaderChecker: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 197, schema size: 21
CSV file: file:///home/user1129/MyDocuments/10_14_Life_Admin/13_Technology/100_Coding/100_01_Python/data_engineering/hard_drive_failure_analytics_dashboard/data/source_csv/2025/data_Q1_2025/2025-03-09.csv
26/03/25 23:45:35 WARN CSVHeaderChecker: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 197, schema size: 21
CSV file: file:///home/user1129/MyDocuments/10_14_Life_Admin/13_Technology/100_Coding/100_01_Python/data_engineering/ha

In [28]:
spark.stop()